# Comparision bw mast3r pixel react and expert trajectory

## Data visualization

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import os
import cv2
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

import subprocess
import sys

# This magic command ensures plots are displayed directly in the Jupyter Notebook
%matplotlib inline

# Optional: Set a clean visual style for our plots
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

In [3]:
# ==============================================================================
# CONFIGURATION BLOCK
# ==============================================================================
df_summary = pd.read_csv('/home/anirudhadg/mast3r-nav-ati/data/speed0_8mps/2026-04-29-10-18-34-pm-v2-15-2nd-board-Ati-bglr-ati-fleet/summary.csv').sort_values('time')
df_tracker = pd.read_csv('/home/anirudhadg/mast3r-nav-ati/data/speed0_8mps/2026-04-29-10-18-34-pm-v2-15-2nd-board-Ati-bglr-ati-fleet/debug/tracker.csv').sort_values('time')
df_drive = pd.read_csv('/home/anirudhadg/mast3r-nav-ati/data/speed0_8mps/2026-04-29-10-18-34-pm-v2-15-2nd-board-Ati-bglr-ati-fleet/drive.csv').sort_values('time')

VIDEO_PATH = '/home/anirudhadg/mast3r-nav-ati/data/speed0_8mps/2026-04-29-10-18-34-pm-v2-15-2nd-board-Ati-bglr-ati-fleet/debug/cam_front0-0-00.mp4'  # Path to your video input file
OUTPUT_DIR = './bev_stepwise_plots'      # Directory where sequential figures are saved

# Control the exact video processing window (in seconds)
# START_TIME = 45.0   # e.g., 0:45
# END_TIME   = 300.0  # e.g., 2:25

# Sensitivity factor for steering arrow deflection based on angular velocity rate
W_SENSITIVITY = 1.0

# Shift camera feed in seconds. Can be positive (+) or negative (-)
CAMERA_OFFSET = 33.0  

RENDER_INTERVAL_SEC = 1.0

MAX_WORKERS = 16

In [4]:
print("Cleaning drive data and synchronizing spatial coordinates...")

# 1. Handle repeating frame_ids: Keep only the first observed value
df_drive = df_drive.drop_duplicates(subset=['frame_id'], keep='first')

# Merge spatial metrics onto the high-frequency drive data points via chronological lookup
df_dense = pd.merge_asof(df_drive, df_summary[['time', 'x', 'y', 'theta']], on='time', direction='nearest')

# Calculate relative elapsed baseline track timeline (seconds from shared run origin)
shared_start_ts = df_dense['time'].iloc[0]
df_dense['elapsed_sec'] = df_dense['time'] - shared_start_ts
df_tracker['elapsed_sec'] = df_tracker['time'] - shared_start_ts

# Determine the true common start: if CAMERA_OFFSET is negative, the video doesn't
# start until telemetry time -CAMERA_OFFSET; if positive/zero, telemetry already covers t=0.
# Whichever stream starts later dictates where the synchronized visualization begins.
EFFECTIVE_START_TIME = max(0.0, -CAMERA_OFFSET)

# Trim only the lead-in before the later-starting stream; no end trim, run to the end of data
df_dense_filtered = df_dense[df_dense['elapsed_sec'] >= EFFECTIVE_START_TIME].copy()
df_tracker_filtered = df_tracker[df_tracker['elapsed_sec'] >= EFFECTIVE_START_TIME].copy()

# Render every frame in the filtered range (no fixed-count sampling)
# Downsample to ~1 row per RENDER_INTERVAL_SEC by bucketing elapsed time and taking
# the first row in each bucket — this thins the *rendered* set only.
# df_dense_filtered / df_tracker_filtered stay at full resolution, so trajectory
# history lines are unaffected and remain smooth/complete.
df_sampled = (
    df_dense_filtered
    .assign(_time_bucket=(df_dense_filtered['elapsed_sec'] // RENDER_INTERVAL_SEC).astype(int))
    .groupby('_time_bucket', as_index=False)
    .first()
    .sort_values('elapsed_sec')
    .drop(columns='_time_bucket')
)

Cleaning drive data and synchronizing spatial coordinates...


In [5]:
# ==============================================================================
# VIDEO DECODING & TEMPORAL SEEDING
# ==============================================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Opening video track: {VIDEO_PATH}")
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"Could not open or locate video file: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video Properties: {fps} FPS, {total_frames} total frames available.")


Opening video track: /home/anirudhadg/mast3r-nav-ati/data/speed0_8mps/2026-04-29-10-18-34-pm-v2-15-2nd-board-Ati-bglr-ati-fleet/debug/cam_front0-0-00.mp4
Video Properties: 5.993443263197029 FPS, 1064 total frames available.


In [ ]:
#BEV Step-Wise generation function


# print(f"Generating plots for all {len(df_sampled)} frames starting at {EFFECTIVE_START_TIME:.2f}s using {MAX_WORKERS} workers...")

# # Thread-local storage so each worker thread gets its own VideoCapture instance
# # (cv2.VideoCapture is not safe to share/seek across threads)
# thread_local = threading.local()

# def get_thread_capture():
#     if not hasattr(thread_local, "cap"):
#         thread_local.cap = cv2.VideoCapture(VIDEO_PATH)
#     return thread_local.cap

# def process_frame(args):
#     plot_index, current_state = args
#     current_time_sec = current_state['elapsed_sec']

#     cap_local = get_thread_capture()

#     # Calculate which video frame matches this specific telemetry frame's time + the camera offset
#     video_time_target = current_time_sec + CAMERA_OFFSET
#     video_frame_idx = int(max(0, video_time_target * fps))

#     cap_local.set(cv2.CAP_PROP_POS_FRAMES, video_frame_idx)
#     ret, frame = cap_local.read()

#     if not ret:
#         print(f"Warning: Could not decode video frame at index {video_frame_idx}. Skipping plot.")
#         return False

#     # Isolate telemetry historical data up to this exact moment to draw the trailing path
#     df_history_dense = df_dense_filtered[df_dense_filtered['elapsed_sec'] <= current_time_sec]
#     df_history_tracker = df_tracker_filtered[df_tracker_filtered['elapsed_sec'] <= current_time_sec]

#     # Build figure via the Figure class directly (thread-safe, no shared pyplot state)
#     fig = Figure(figsize=(20, 10))
#     ax_map, ax_img = fig.subplots(1, 2, gridspec_kw={'width_ratios': [1.2, 1.0]})

#     # --------------------------------------------------------------------------
#     # LEFT PANE: Bird's Eye View Spatial Trajectory
#     # --------------------------------------------------------------------------
#     ax_map.plot(df_tracker_filtered['ref_x'], df_tracker_filtered['ref_y'],
#                 color='gray', linestyle=':', alpha=0.4, label='Full Planned Route Window')
#     ax_map.plot(df_dense_filtered['x'], df_dense_filtered['y'],
#                 color='royalblue', linestyle='-', alpha=0.2, label='Full Actual Route Window')

#     if len(df_history_tracker) > 0:
#         ax_map.plot(df_history_tracker['ref_x'], df_history_tracker['ref_y'],
#                     color='black', linestyle='--', linewidth=1.5, label='Planned Trajectory (To Current)')
#     ax_map.plot(df_history_dense['x'], df_history_dense['y'],
#                 color='royalblue', linewidth=2.5, label='Traveled Trajectory (To Current)')

#     steered_angle = current_state['theta'] + (current_state['angular_velocity'] * W_SENSITIVITY)
#     u_vec = current_state['linear_velocity'] * np.cos(steered_angle)
#     v_vec = current_state['linear_velocity'] * np.sin(steered_angle)

#     ax_map.quiver(current_state['x'], current_state['y'], u_vec, v_vec,
#                   color='red', scale=10, width=0.008, headwidth=4, zorder=5,
#                   label=f"Frame ID: {int(current_state['frame_id'])}\n(v={current_state['linear_velocity']:.2f}, w={current_state['angular_velocity']:.2f})")

#     ax_map.scatter(current_state['x'], current_state['y'], color='red', s=100, edgecolors='black', zorder=6)

#     ax_map.set_title(f"BEV Trajectory Map (Telemetry Clock: {current_time_sec:.2f}s)", fontsize=14, fontweight='bold')
#     ax_map.set_xlabel("Global X Position (meters)", fontsize=11)
#     ax_map.set_ylabel("Global Y Position (meters)", fontsize=11)
#     ax_map.axis('equal')
#     ax_map.legend(loc='upper left', frameon=True, facecolor='white', framealpha=0.9)

#     # --------------------------------------------------------------------------
#     # RIGHT PANE: Corresponding Egoview Video Frame
#     # --------------------------------------------------------------------------
#     frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#     ax_img.imshow(frame_rgb)
#     ax_img.set_title(f"Video Feed View: Frame ID {int(current_state['frame_id'])} (Video Clock: {video_time_target:.1f}s)", fontsize=14, fontweight='bold')
#     ax_img.axis('off')

#     # --------------------------------------------------------------------------
#     # SAVE FIGURE DISK PIPELINE
#     # --------------------------------------------------------------------------
#     fig.tight_layout()
#     output_filename = os.path.join(OUTPUT_DIR, f"step_{plot_index:05d}.png")
#     fig.savefig(output_filename, dpi=100, bbox_inches='tight')

#     return True

# # Preserve original ordering by pairing each row with its output index up front
# work_items = list(enumerate(row for _, row in df_sampled.iterrows()))

# saved_plot_count = 0
# with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
#     futures = [executor.submit(process_frame, item) for item in work_items]
#     for future in as_completed(futures):
#         if future.result():
#             saved_plot_count += 1

# print(f"Process finalized successfully! Generated {saved_plot_count} sync-plots stored in directory: '{OUTPUT_DIR}'")

In [14]:
# ==============================================================================
# SAVE 200 SEQUENTIAL RAW VIDEO FRAMES (DOWNSAMPLED TO MAPPER CONFIG)
# ==============================================================================
from omegaconf import OmegaConf

FRAME_SAVE_DIR = '/home/anirudhadg/mast3r-nav-ati/data/mast3r-nav/data/recordings/traj_001/images'
NUM_FRAMES_TO_SAVE = 200
os.makedirs(FRAME_SAVE_DIR, exist_ok=True)


try:
    cfg_map = OmegaConf.load("/home/anirudhadg/mast3r-nav-ati/mast3r-nav/configs/mapper/mapper_config.yaml")

    target_w = cfg_map.image.width
    target_h = cfg_map.image.height

# 1. Fetch target resolution from mapper config (with standard fallback keys)
# try:
#     if "cfg" in globals():
#         target_w = cfg.mapper.img_w
#         target_h = cfg.mapper.img_h
#     else:
#         # Load directly if cfg isn't already loaded in notebook state
#         cfg_map = OmegaConf.load("configs/mapper/create_topomap.yaml")
#         target_w = cfg_map.img_w
#         target_h = cfg_map.img_h
except Exception as e:
    print(f"Warning: Could not read mapper config ({e}). Defaulting to 512x512.")
    target_w, target_h = 512, 512

print(f"Target frame resolution set to: {target_w}x{target_h}")

# 2. Setup frame sampling indices
video_start_frame_idx = int(max(0, CAMERA_OFFSET * fps))
video_end_frame_idx = total_frames - 1

if video_start_frame_idx >= video_end_frame_idx:
    raise ValueError(f"CAMERA_OFFSET ({CAMERA_OFFSET}s) leaves no video range to sample from.")

frame_indices = np.linspace(video_start_frame_idx, video_end_frame_idx, NUM_FRAMES_TO_SAVE).astype(int)

# 3. Extract, resize, and save frames
cap_save = cv2.VideoCapture(VIDEO_PATH)
if not cap_save.isOpened():
    raise FileNotFoundError(f"Could not open or locate video file: {VIDEO_PATH}")

saved_count = 0
for i, frame_idx in enumerate(frame_indices):
    cap_save.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ret, frame = cap_save.read()
    if not ret:
        print(f"Warning: could not decode frame at index {frame_idx}. Skipping.")
        continue
    
    # Downsample frame to match mapper model expected dimensions
    resized_frame = cv2.resize(frame, (target_w, target_h), interpolation=cv2.INTER_AREA)
    
    output_path = os.path.join(FRAME_SAVE_DIR, f"frame_{i:05d}.jpg")
    cv2.imwrite(output_path, resized_frame)
    saved_count += 1

cap_save.release()
print(f"Saved {saved_count} downsampled frames ({target_w}x{target_h}) from frame "
      f"{video_start_frame_idx} to {video_end_frame_idx} ({CAMERA_OFFSET:.2f}s to end) to '{FRAME_SAVE_DIR}'")

Target frame resolution set to: 320x240
Saved 200 downsampled frames (320x240) from frame 197 to 1063 (33.00s to end) to '/home/anirudhadg/mast3r-nav-ati/data/mast3r-nav/data/recordings/traj_001/images'


## Running the mapper

In [15]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "libs.mapper.create_topomap"],
    check=True
)


TOPOLOGICAL MAP CREATION

Using configuration from: {'image': {'width': 320, 'height': 240}, 'model': {'name': 'mast3r', 'path': '/home/anirudhadg/mast3r-nav-ati/mast3r-nav/checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth', 'device': 'cuda', 'subsample_or_initxy1': 8}, 'graph': {'inter_image_match_window_size': 3, 'edge_culling_mode': 'NONE', 'node_culling_mode': 'NONE', 'node_culling_factor': 10}, 'processing': {'subsample_start_idx': 0, 'subsample_end_idx': None, 'subsample_step': 1, 'force_recompute_graph': True, 'force_recompute_masks': False, 'copy_images': False}, 'compression': {'enabled': True, 'chunk_size': 1073741824}, 'scenes': {'base_dir': '/home/anirudhadg/mast3r-nav-ati/data/mast3r-nav/data/recordings', 'multi_scene': False, 'base_out_dir': '/home/anirudhadg/mast3r-nav-ati/data/mast3r-nav/data/recordings/maps', 'scene_name': 'traj_001', 'scene_list_file': '/home/onyx/work_dirs/vanshg/navigation/mast3r-nav/episodes_removing_blacklist.txt', 'start_idx': 0, 

Processing scenes:   0%|          | 0/1 [00:00<?, ?scene/s]

instantiating : AsymmetricMASt3R(enc_depth=24, dec_depth=12, enc_embed_dim=1024, dec_embed_dim=768, enc_num_heads=16, dec_num_heads=12, pos_embed='RoPE100',img_size=(512, 512), head_type='catmlp+dpt', output_mode='pts3d+desc24', depth_mode=('exp', -inf, inf), conf_mode=('exp', 1, inf), patch_embed_cls='PatchEmbedDust3R', two_confs=True, desc_conf_mode=('exp', 0, inf), landscape_only=False)
<All keys matched successfully>
[2026-07-22 12:36:45,787][libs.mast3r_utils.model][INFO] - MASt3R model loaded successfully

Mode: CONFIG - Creating topological map...



Computing MASt3R point clouds:   0%|          | 0/200 [00:00<?, ?it/s]/home/anirudhadg/mast3r-nav-ati/mast3r-nav/libs/matcher/mast3r/dust3r/dust3r/inference.py:44: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=bool(use_amp)):
/home/anirudhadg/mast3r-nav-ati/mast3r-nav/libs/matcher/mast3r/dust3r/dust3r/model.py:206: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/home/anirudhadg/mast3r-nav-ati/mast3r-nav/libs/matcher/mast3r/dust3r/dust3r/inference.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):

Computing MASt3R point clouds: 100%|██████████| 200/200 [00:11<00:00, 17.18it/s]


First pass: collecting all match pairs...


...

Final graph has 73336 nodes and 14120614 edges
✓ Created topological map in 100.70s
  Graph: 73336 nodes, 14120614 edges

Saving base graph...
Processing 73336 nodes...
Processing 14120614 edges...
Data preparation took: 23.208s
Saved to: /home/anirudhadg/mast3r-nav-ati/data/mast3r-nav/data/recordings/maps/traj_001/graph_base_320x240_EC_NONE_NC_NONE_NCF_10.pkl.b2s | Total Time: 29.844s

Computing distances to goal node...
Goal: img_idx=102, pixel=(220, 110)
Added goal node 7869020 at pixel (220, 110) in image 102
Found 366 DA nodes in image 102
Connected goal node 7869020 to 366 DA nodes

Computing distance-to-goal costmaps for all images...



computing non-da to goal distances: 100%|██████████| 200/200 [00:04<00:00, 45.84it/s]


✓ Saved costmaps to /home/anirudhadg/mast3r-nav-ati/data/mast3r-nav/data/recordings/maps/traj_001/costmaps_320x240_EC_NONE_NC_NONE_NCF_10
✓ Computed distances in 18.52s

Saving goal graph...
Processing 73337 nodes...
Processing 14120980 edges...
Converted 73337 path lengths to arrays
Data preparation took: 14.983s
Saved to: /home/anirudhadg/mast3r-nav-ati/data/mast3r-nav/data/recordings/maps/traj_001/graph_with_distances_to_goal_320x240_EC_NONE_NC_NONE_NCF_10.pkl.b2s | Total Time: 21.299s

✓ SCENE COMPLETE in 174.36s


✓ COMPLETE: 1/1 scenes processed successfully


Processing scenes: 100%|██████████| 1/1 [02:54<00:00, 174.37s/scene]


CompletedProcess(args=['/home/anirudhadg/mast3r-nav-ati/mast3r-nav/.pixi/envs/default/bin/python', '-m', 'libs.mapper.create_topomap'], returncode=0)

run 
``` pixi run python -m libs.mapper.create_topomap ```
to make the map and train the controller

## Model

### Loading the model for waypointing

In [16]:
import sys
import math
import torch
import torchvision.transforms as tfm
from PIL import Image
from omegaconf import OmegaConf
from hydra import initialize, compose

# Import MASt3R-Nav blocks
from libs.mapper.create_topomap import CostmapData
from libs.matcher.mast3r_matcher import Mast3rMatcher
from libs.localizer.loc_topo import LocalizeTopological
from libs.planner.plan_topo import PlanTopological
from libs.goal_generator.goal_gen import GoalGenerator
from libs.common.utils_sim import build_intrinsics
from libs.control.learnt_controller import ObjRelLearntController
from dust3r.inference import inference

import time
if torch.cuda.is_available():
    torch.cuda.synchronize()
    _gpu_before = torch.cuda.memory_allocated() / 1024**2
_load_t0 = time.perf_counter()

# Import notebook utilities
sys.path.append("notebooks")
import viz_utils

def get_pts3d_direct(matcher, rgb_img, device):
    """Memory-optimized forward pass extracting 3D points."""
    normalize = tfm.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    img_pil = Image.fromarray(rgb_img)
    img_tensor = tfm.ToTensor()(img_pil)
    img_tensor = tfm.Resize((matcher.resize_h, matcher.resize_w), antialias=True)(img_tensor)
    img_tensor = normalize(img_tensor).unsqueeze(0).to(device)
    
    img_pair = [
        {"img": img_tensor, "idx": 0, "instance": 0, "true_shape": np.int32([img_tensor.shape[-2:]])},
        {"img": img_tensor.clone(), "idx": 1, "instance": 1, "true_shape": np.int32([img_tensor.shape[-2:]])},
    ]
    with torch.no_grad():
        output = inference([tuple(img_pair)], matcher.model, device, batch_size=1, verbose=False)
    return output["pred1"]["pts3d"][0].cpu().numpy()

print("Loading configurations...")
# Clear hydra state if rerunning cell
from hydra.core.global_hydra import GlobalHydra
GlobalHydra.instance().clear()

with initialize(version_base=None, config_path="configs"):
    cfg = compose(config_name="config")

ctrl_cfg = OmegaConf.load("configs/controller/gnm_waypixel.yaml")
ctrl_cfg.load_run = "checkpoints/gnm_mast3r_nav"
device = "cuda" if torch.cuda.is_available() else "cpu"

img_w, img_h = int(cfg.matcher.resize_w), int(cfg.matcher.resize_h)

print("Loading map...")
# IMPORTANT: Adjust this to the actual map npz you are evaluating against
costmap_path = "/home/anirudhadg/mast3r-nav-ati/data/mast3r-nav/data/recordings/maps/traj_001/costmaps_320x240_EC_NONE_NC_NONE_NCF_10.npz"
costmap_data = CostmapData.from_npz(costmap_path)
map_img_paths = costmap_data.get_metadata()["image_paths"]

print("Loading Neural Networks into VRAM (This takes a moment)...")
matcher = Mast3rMatcher(resize_w=img_w, resize_h=img_h, device=device)
localizer = LocalizeTopological(map_img_paths, img_h, img_w, matcher, cfg.localizer)
planner = PlanTopological(img_h, img_w, costmap_data, device, cfg.planner)
goal_generator = GoalGenerator(img_h, img_w, localizer, planner, cfg)

intrinsics = build_intrinsics(image_width=img_w, image_height=img_h, 
                              field_of_view_radians_u=math.radians(79), device=device)

controller = ObjRelLearntController(
    config=OmegaConf.to_container(ctrl_cfg, resolve=True),
    goal_source=cfg.goal_source,
    boost_final_goal=cfg.get("boost_final_goal", False),
)

print("✅ Models Loaded Successfully!")

if torch.cuda.is_available():
    torch.cuda.synchronize()
    _gpu_after = torch.cuda.memory_allocated() / 1024**2
    print(f"Model GPU footprint: {_gpu_after - _gpu_before:.1f} MB "
          f"(total torch-allocated now: {_gpu_after:.1f} MB)")
print(f"Model load time: {time.perf_counter() - _load_t0:.2f} s")

Loading configurations...
Loading map...
Loading Neural Networks into VRAM (This takes a moment)...
... loading model from /home/anirudhadg/mast3r-nav-ati/mast3r-nav/libs/matcher/mast3r/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth
instantiating : AsymmetricMASt3R(enc_depth=24, dec_depth=12, enc_embed_dim=1024, dec_embed_dim=768, enc_num_heads=16, dec_num_heads=12, pos_embed='RoPE100',img_size=(512, 512), head_type='catmlp+dpt', output_mode='pts3d+desc24', depth_mode=('exp', -inf, inf), conf_mode=('exp', 1, inf), patch_embed_cls='PatchEmbedDust3R', two_confs=True, desc_conf_mode=('exp', 0, inf), landscape_only=False)
<All keys matched successfully>
Loading model from  checkpoints/gnm_mast3r_nav
SUCESSFULLY LOADED MODEL FROM latest_path = 'checkpoints/gnm_mast3r_nav/latest.pth'
Learnt controller initialized!
✅ Models Loaded Successfully!
Model GPU footprint: -1.2 MB (total torch-allocated now: 2641.2 MB)
Model load time: 3.59 s


#### localzation sanity check

In [ ]:
# ==============================================================================
# LOCALIZATION SANITY CHECK  (+ compare timing & per-process GPU stats)
# ==============================================================================
import time

QUERY_IMAGE_PATH = '/home/anirudhadg/mast3r-nav-ati/data/mast3r-nav/data/recordings/traj_001/frame_00059.png'

use_cuda = torch.cuda.is_available()

def query_process_gpu_mem_mb():
    """GPU memory (MB) used by THIS process only, via nvidia-smi (includes CUDA context)."""
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-compute-apps=pid,used_memory',
             '--format=csv,noheader,nounits'], encoding='utf-8')
    except Exception:
        return None
    for line in out.strip().splitlines():
        parts = [p.strip() for p in line.split(',')]
        if len(parts) >= 2 and parts[0].isdigit() and int(parts[0]) == os.getpid():
            return float(parts[1])
    return None

# --- Load + prep the query image ----------------------------------------------
frame = cv2.imread(QUERY_IMAGE_PATH)
assert frame is not None, f"Could not read {QUERY_IMAGE_PATH}"
qry_rgb = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (img_w, img_h))

# --- Reset localizer state so this call stands alone --------------------------
localizer.matched_img_history = []
localizer.ref_imgs_tried = np.array([])
localizer.reloc_dia = localizer.reloc_dia_default
localizer.reloc_exhausted = False
localizer.lost = False

# --- GPU baseline + peak reset before the compare -----------------------------
if use_cuda:
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

# --- Timed localization against the WHOLE map ---------------------------------
candidate_indices = np.arange(len(map_img_paths))
if use_cuda:
    torch.cuda.synchronize()
t0 = time.perf_counter()

qry_mkpts, ref_mkpts, confidences = localizer.localize(qry_rgb, candidate_indices)

if use_cuda:
    torch.cuda.synchronize()          # make sure GPU work is done before we stop the clock
elapsed = time.perf_counter() - t0

if len(ref_mkpts) == 0:
    raise RuntimeError("Localization failed: no matches found for this image.")

localized_idx = int(localizer.localized_img_idx)
counts = np.bincount(ref_mkpts[:, 0].astype(int), minlength=len(map_img_paths))
n_cand = len(candidate_indices)

# --- Stats printout -----------------------------------------------------------
print("=" * 62)
print(f"Localized index : {localized_idx}   ({counts[localized_idx]} matched keypoints)")
top5 = np.argsort(counts)[::-1][:5]
print("Top-5 candidates:", [(int(i), int(counts[i])) for i in top5])
print("-" * 62)
print(f"Compare time    : {elapsed:.3f} s over {n_cand} map images "
      f"({elapsed / n_cand * 1000:.1f} ms/image)")
if use_cuda:
    print("-" * 62)
    print("GPU (torch allocator, this process):")
    print(f"  allocated now        : {torch.cuda.memory_allocated() / 1024**2:8.1f} MB")
    print(f"  peak during localize : {torch.cuda.max_memory_allocated() / 1024**2:8.1f} MB")
    print(f"  reserved (cached)    : {torch.cuda.memory_reserved() / 1024**2:8.1f} MB")
    proc_mem = query_process_gpu_mem_mb()
    if proc_mem is not None:
        print(f"  process total (nvidia-smi, incl. CUDA ctx): {proc_mem:8.1f} MB")
print("=" * 62)

# --- Load matched map image + plot side by side -------------------------------
map_bgr = cv2.imread(str(map_img_paths[localized_idx]))
assert map_bgr is not None, f"Could not read map image {map_img_paths[localized_idx]}"
map_rgb = cv2.cvtColor(map_bgr, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(qry_rgb)
axes[0].set_title(f"Query\n{os.path.basename(QUERY_IMAGE_PATH)}", fontsize=13)
axes[0].axis('off')
axes[1].imshow(map_rgb)
axes[1].set_title(f"Localized Map idx {localized_idx}  |  {counts[localized_idx]} matches", fontsize=13)
axes[1].axis('off')
fig.suptitle(f"Localization: {elapsed:.3f}s over {n_cand} images", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# SEQUENTIAL MODEL INFERENCE
#   ABSOLUTE goal test: localized node's PRECOMPUTED median cost-to-goal
#   (read from the .npz, not a frame-to-frame delta) + topological arrival.
#   Records EVERY step. Dumps metrics + config to info.txt.
# ==============================================================================
import glob, time, subprocess, datetime
from omegaconf import OmegaConf

IMAGE_DIR = '/home/anirudhadg/mast3r-nav-ati/data/mast3r-nav/data/recordings/traj_001/images/'
SAVE_DIR  = IMAGE_DIR + "bev_images"
SEED_SUBSAMPLE = 1
CLEAR_HISTORY_AFTER_SEED = True

# --- absolute goal-reached knobs ----------------------------------------------
COST_SENTINEL     = 99.0    # invalid/unreachable pixel value
GOAL_COST_THRESH  = 3.0     # localized-node median cost-to-goal (path-len units) => at goal
GOAL_IDX_TOL      = 5       # localized index within this of goal_img_idx => at goal
MIN_PROGRESS_FRAC = 0.25    # must have advanced past this fraction of the map first
CONFIRM_FRAMES    = 3       # goal condition must hold this many consecutive frames

use_cuda = torch.cuda.is_available()
def gpu_total_used_mb():
    try:
        o=subprocess.check_output(['nvidia-smi','--query-gpu=memory.used,memory.total',
          '--format=csv,noheader,nounits'],encoding='utf-8').strip().splitlines()[0]
        u,t=[float(x) for x in o.split(',')]; return u,t
    except Exception: return None,None
def gpu_proc_used_mb():
    try:
        o=subprocess.check_output(['nvidia-smi','--query-compute-apps=pid,used_memory',
          '--format=csv,noheader,nounits'],encoding='utf-8')
        for ln in o.strip().splitlines():
            p=[x.strip() for x in ln.split(',')]
            if len(p)>=2 and p[0].isdigit() and int(p[0])==os.getpid(): return float(p[1])
    except Exception: pass
    return None

# --- precomputed per-node cost-to-goal (absolute, from the map file) ----------
img_costmaps = costmap_data.get_costmap()
def node_cost_to_goal(idx):
    cm = np.asarray(img_costmaps[idx], dtype=float)
    cm = cm[np.isfinite(cm) & (cm < COST_SENTINEL)]
    return float(np.median(cm)) if cm.size else np.inf

try:    goal_img_idx = int(costmap_data.get_metadata()['goal_img_idx'])
except Exception: goal_img_idx = len(map_img_paths) - 1
n_map = len(map_img_paths)
goal_node_cost = node_cost_to_goal(goal_img_idx)
print(f"Goal node = {goal_img_idx}/{n_map-1} | its median cost-to-goal = {goal_node_cost:.2f}")

# --- images + telemetry -------------------------------------------------------
image_paths=[]
for ext in ('*.png','*.jpg','*.jpeg'): image_paths+=glob.glob(os.path.join(IMAGE_DIR,ext))
image_paths=sorted(image_paths); print(f"{len(image_paths)} images in {IMAGE_DIR}")

def image_timestamps(n):
    v0=int(max(0,CAMERA_OFFSET*fps)); return np.linspace(v0,total_frames-1,n).astype(int)/fps-CAMERA_OFFSET
elapsed=image_timestamps(len(image_paths))
tel_t=df_dense_filtered['elapsed_sec'].to_numpy(); tel_x=df_dense_filtered['x'].to_numpy()
tel_y=df_dense_filtered['y'].to_numpy(); tel_theta=df_dense_filtered['theta'].to_numpy()
tel_vbot=df_dense_filtered['linear_velocity'].to_numpy(); tel_wbot=df_dense_filtered['angular_velocity'].to_numpy()
def nearest_row(t):
    j=int(np.argmin(np.abs(tel_t-t))); return tel_x[j],tel_y[j],tel_theta[j],tel_vbot[j],tel_wbot[j]
def map_idx_to_xy(idx):
    frac=0.0 if n_map<=1 else np.clip(idx/(n_map-1),0,1)
    r=df_dense_filtered.iloc[int(round(frac*(len(df_dense_filtered)-1)))]; return float(r['x']),float(r['y'])
goal_xy=map_idx_to_xy(goal_img_idx)

# --- reset localizer + controller ONCE ----------------------------------------
controller.reset_params()
localizer.matched_img_history=[]; localizer.ref_imgs_tried=np.array([])
localizer.reloc_dia=localizer.reloc_dia_default; localizer.reloc_exhausted=False
localizer.lost=False; localizer.localizer_iter_lb=0; localizer.localized_img_idx=0

buf={k:[] for k in ('x','y','theta','v_bot','w_bot','v_model','w_model','loc_idx','node_cost','wp')}
frame_times=[]; stop_frame=None; reached_reason=None; confirm=0
if use_cuda: torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()

for i,p in enumerate(image_paths):
    frame=cv2.imread(p)
    if frame is None: print(f"  [skip] {p}"); continue
    rgb=cv2.resize(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB),(img_w,img_h))
    candidates=np.arange(0,n_map,SEED_SUBSAMPLE) if i==0 else None

    if use_cuda: torch.cuda.synchronize()
    t0=time.perf_counter()
    qry_pts3d=get_pts3d_direct(matcher,rgb,device)
    costmap=goal_generator.get_goal_mask(qry_img=rgb,qry_depth=None,qry_pts3d=qry_pts3d,
            intrinsics=intrinsics,candidate_img_indices=candidates,return_vis_data=False)
    if costmap is None: print(f"  [skip] loc failed {os.path.basename(p)}"); continue
    if i==0 and CLEAR_HISTORY_AFTER_SEED: localizer.matched_img_history=[]
    v,w,_=controller.predict(rgb,costmap)
    buf['wp'].append(controller.action_pred.copy())
    if use_cuda: torch.cuda.synchronize()
    frame_times.append(time.perf_counter()-t0)

    loc=int(localizer.localized_img_idx); ncost=node_cost_to_goal(loc)
    x,y,theta,v_bot,w_bot=nearest_row(elapsed[i])
    for k,val in zip(buf,(x,y,theta,v_bot,w_bot,v,w,loc,ncost)): buf[k].append(val)

    if i%20==0:
        print(f"  [{i:3d}] loc={loc:3d}/{goal_img_idx} node_cost={ncost:6.2f} "
              f"| bot(v={v_bot:.3f},w={w_bot:.3f}) model(v={v:.3f},w={w:.3f}) | {frame_times[-1]*1000:.0f}ms")

    # ---- ABSOLUTE goal test: topological arrival ONLY (index-based) ----------
    # node_cost median test removed: it dipped under a blind threshold long
    # before the goal and stopped the run early. loc_idx reaching the goal
    # node is the reliable signal; keep node_cost only for logging/plots.
    if loc >= goal_img_idx - GOAL_IDX_TOL:
        confirm += 1
    else:
        confirm = 0
    if confirm >= CONFIRM_FRAMES:
        stop_frame = i
        reached_reason = f"localized index {loc} reached goal node {goal_img_idx} (held {CONFIRM_FRAMES} frames)"
        break

seq={k:(np.array(v) if k!='wp' else v) for k,v in buf.items()}   # keep wp as a list of arrays
print(f"\nRecorded {len(seq['x'])} steps. "
      f"{'GOAL REACHED: '+reached_reason if stop_frame is not None else 'ran out of frames before goal.'}")

# --- METRICS ------------------------------------------------------------------
ft=np.array(frame_times); lines=[]
def L(s): lines.append(s); print(s)
L("="*62)
L(f"Run: {datetime.datetime.now().isoformat(timespec='seconds')}")
L(f"IMAGE_DIR: {IMAGE_DIR}")
L(f"Frames run: {len(ft)} | stopped early: {stop_frame is not None} | reason: {reached_reason}")
L(f"Goal: node={goal_img_idx}/{n_map-1}  BEV(x,y)={goal_xy}  goal_node_cost={goal_node_cost:.2f}")
if len(ft)>1:
    st=ft[1:]
    L(f"per-frame time  frame0/global-seed : {ft[0]*1000:.1f} ms")
    L(f"per-frame time  windowed mean/med/p90/max : "
      f"{st.mean()*1000:.1f}/{np.median(st)*1000:.1f}/{np.percentile(st,90)*1000:.1f}/{st.max()*1000:.1f} ms")
    L(f"throughput (windowed) : {1/st.mean():.2f} FPS")
if use_cuda:
    proc,(tu,tc)=gpu_proc_used_mb(),gpu_total_used_mb()
    L(f"GPU torch alloc/peak/reserved : {torch.cuda.memory_allocated()/1024**2:.0f}/"
      f"{torch.cuda.max_memory_allocated()/1024**2:.0f}/{torch.cuda.memory_reserved()/1024**2:.0f} MB")
    if proc is not None: L(f"GPU this process (nvidia-smi) : {proc:.0f} MB")
    if tu   is not None: L(f"GPU total used/capacity       : {tu:.0f}/{tc:.0f} MB")
L(f"stop knobs: GOAL_COST_THRESH={GOAL_COST_THRESH} GOAL_IDX_TOL={GOAL_IDX_TOL} "
  f"MIN_PROGRESS_FRAC={MIN_PROGRESS_FRAC} CONFIRM_FRAMES={CONFIRM_FRAMES}")
L("="*62)

# --- info.txt (metrics + full config) -----------------------------------------
os.makedirs(SAVE_DIR, exist_ok=True)
info_path=os.path.join(SAVE_DIR,'info.txt')
with open(info_path,'w') as f:
    f.write("MASt3R-Nav shadow-mode run\n")
    f.write("\n".join(lines)+"\n\n")
    f.write("bot   v[%.3f,%.3f] w[%.3f,%.3f]\n"%(seq['v_bot'].min(),seq['v_bot'].max(),
                                                 seq['w_bot'].min(),seq['w_bot'].max()))
    f.write("model v[%.3f,%.3f] w[%.3f,%.3f]\n"%(seq['v_model'].min(),seq['v_model'].max(),
                                                 seq['w_model'].min(),seq['w_model'].max()))
    f.write(f"localized index path: {seq['loc_idx'].tolist()}\n\n")
    f.write("="*62+"\nFULL CONFIG\n"+"="*62+"\n")
    try:    f.write(OmegaConf.to_yaml(cfg))
    except Exception as e: f.write(f"(could not dump cfg: {e})\n")
print(f"Wrote {info_path}")

Goal node = 102/199 | its median cost-to-goal = 0.66
200 images in /home/anirudhadg/mast3r-nav-ati/data/mast3r-nav/data/recordings/traj_001/images/


No candidate indices provided, Auto-Genering candidate map img indices


  [  0] loc= 11/102 node_cost= 14.90 | bot(v=0.000,w=0.000) model(v=0.036,w=-0.100) | 11281ms


No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No can

  [ 20] loc= 10/102 node_cost= 14.93 | bot(v=0.258,w=-0.002) model(v=0.042,w=-0.100) | 474ms


...

  [160] loc=  8/102 node_cost= 14.93 | bot(v=0.275,w=-0.006) model(v=0.050,w=-0.009) | 453ms


No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No can

  [180] loc=  6/102 node_cost= 14.92 | bot(v=0.394,w=0.004) model(v=0.050,w=0.004) | 466ms


No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No candidate indices provided, Auto-Genering candidate map img indices
No can


Recorded 200 steps. ran out of frames before goal.
Run: 2026-07-22T12:57:33
IMAGE_DIR: /home/anirudhadg/mast3r-nav-ati/data/mast3r-nav/data/recordings/traj_001/images/
Frames run: 200 | stopped early: False | reason: None
Goal: node=102/199  BEV(x,y)=(-0.3696031607061243, 16.314449982877868)  goal_node_cost=0.66
per-frame time  frame0/global-seed : 11281.2 ms
per-frame time  windowed mean/med/p90/max : 478.6/465.0/527.1/1110.0 ms
throughput (windowed) : 2.09 FPS
GPU torch alloc/peak/reserved : 2668/14762/3910 MB
GPU this process (nvidia-smi) : 4420 MB
GPU total used/capacity       : 4664/49140 MB
stop knobs: GOAL_COST_THRESH=3.0 GOAL_IDX_TOL=5 MIN_PROGRESS_FRAC=0.25 CONFIRM_FRAMES=3
Wrote /home/anirudhadg/mast3r-nav-ati/data/mast3r-nav/data/recordings/traj_001/images//bev_images/info.txt


In [23]:
# ==============================================================================
# THREE BEV GRAPHS + DEBUG  — model as action_pred rollouts (local->global)
# ==============================================================================
W_SENSITIVITY  = 1.0
ARROW_STRIDE   = 1
ROLLOUT_STRIDE = 1
WAYPOINT_SCALE = 0.5
BOT_COLOR, MODEL_COLOR, GOAL_COLOR = 'red', 'blue', 'black'

xs, ys, th = seq['x'], seq['y'], seq['theta']
vb, wb     = seq['v_bot'], seq['w_bot']

def bot_comps(v, w):
    a = th + w*W_SENSITIVITY;  return v*np.cos(a), v*np.sin(a)

def local_to_global(wp, X, Y, theta):
    wp = np.asarray(wp); c, s = np.cos(theta), np.sin(theta)
    gx = X + WAYPOINT_SCALE*(wp[:,0]*c - wp[:,1]*s)
    gy = Y + WAYPOINT_SCALE*(wp[:,0]*s + wp[:,1]*c)
    return np.insert(gx, 0, X), np.insert(gy, 0, Y)

# ============================ DEBUG =========================================
n  = len(xs)
li = seq['loc_idx']
gct = globals().get('GOAL_COST_THRESH'); git = globals().get('GOAL_IDX_TOL')
mpf = globals().get('MIN_PROGRESS_FRAC'); cnf = globals().get('CONFIRM_FRAMES')
print("="*60)
print(f"[DBG] recorded steps in seq            : {n}")
print(f"[DBG] total images available (IMAGE_DIR): {len(image_paths)}")
print(f"[DBG] goal_img_idx={goal_img_idx}  goal_xy={goal_xy}")
print(f"[DBG] stop knobs: GOAL_COST_THRESH={gct} GOAL_IDX_TOL={git} "
      f"MIN_PROGRESS_FRAC={mpf} CONFIRM_FRAMES={cnf}")
print(f"[DBG] loc_idx  first={li[0]} last={li[-1]} min={li.min()} max={li.max()}")
print(f"[DBG] map coverage: max_loc {li.max()}/{goal_img_idx} = {li.max()/max(goal_img_idx,1):.2f}")
print(f"[DBG] loc_idx full path: {li.tolist()}")

if 'node_cost' in seq:
    nc = seq['node_cost']
    print(f"[DBG] node_cost first={nc[0]:.2f} last={nc[-1]:.2f} min={nc.min():.2f} max={nc.max():.2f}")
    if gct is not None:
        hit = nc <= gct
        first_hit = int(np.argmax(hit)) if hit.any() else None
        print(f"[DBG] frames with node_cost<=thr : {hit.sum()}/{n}  first at frame {first_hit}"
              + (f" (loc={li[first_hit]}, {li[first_hit]/max(goal_img_idx,1):.0%} along)" if first_hit is not None else ""))

# Which condition ended Cell A?
idx_reached  = (git is not None) and (li[-1] >= goal_img_idx - git)
cost_reached = ('node_cost' in seq) and (gct is not None) and (seq['node_cost'][-1] <= gct)
if n < len(image_paths):
    culprit = ("INDEX reached goal" if idx_reached else
               "NODE_COST under threshold (likely premature — scale too high)" if cost_reached else
               "unknown / localization jump")
    print(f"[DBG] Cell A STOPPED EARLY at frame {n-1}/{len(image_paths)-1}.  Trigger: {culprit}")
else:
    print("[DBG] Cell A ran the full directory (no early stop).")

n_roll = len(range(0, n, ROLLOUT_STRIDE))
print(f"[DBG] model rollouts to draw           : {n_roll} (ROLLOUT_STRIDE={ROLLOUT_STRIDE})")
for i in list(range(0, n, max(1, n//5)))[:5]:
    wp = np.asarray(seq['wp'][i])
    gx, gy = local_to_global(wp, xs[i], ys[i], th[i])
    disp = np.hypot(gx[-1]-xs[i], gy[-1]-ys[i])
    print(f"[DBG]   frame {i:3d}: wp{wp.shape} |xy|max={np.abs(wp[:,:2]).max():.3f} "
          f"-> global endpoint disp={disp:.3f} m  finite={np.isfinite(wp).all()}")
print("="*60)
# ==========================================================================

def base(ax, title):
    ax.plot(df_dense_filtered['x'], df_dense_filtered['y'], color='gray', lw=1.5,
            alpha=0.35, label='Actual trajectory', zorder=1)
    ax.scatter(*goal_xy, marker='X', s=320, color=GOAL_COLOR, edgecolors='white',
               linewidths=1.5, zorder=10, label=f'Goal (node {goal_img_idx})')
    # mark where the recorded model data actually ends
    ax.scatter(xs[-1], ys[-1], s=140, facecolors='none', edgecolors=MODEL_COLOR,
               linewidths=2, zorder=9, label=f'Last recorded frame ({n-1})')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel("Global X (m)"); ax.set_ylabel("Global Y (m)"); ax.axis('equal')

def draw_bot(ax):
    s = slice(None, None, ARROW_STRIDE); u, vv = bot_comps(vb, wb)
    ax.quiver(xs[s], ys[s], u[s], vv[s], angles='xy', scale_units='xy',
              scale=max(vb.max(),1e-6)/(0.06*max(df_dense_filtered['x'].max()-df_dense_filtered['x'].min(),
                                                 df_dense_filtered['y'].max()-df_dense_filtered['y'].min())),
              width=0.005, headwidth=4, color=BOT_COLOR, zorder=4,
              label=f'Bot v/w (v̄={vb.mean():.3f}, w̄={wb.mean():.3f})')

def draw_model(ax):
    lab = True
    for i in range(0, n, ROLLOUT_STRIDE):
        gx, gy = local_to_global(seq['wp'][i], xs[i], ys[i], th[i])
        ax.plot(gx, gy, color=MODEL_COLOR, alpha=0.55, lw=1.2, zorder=5,
                label='Model waypoint rollout' if lab else None)
        ax.scatter(gx[-1], gy[-1], s=10, color=MODEL_COLOR, zorder=6); lab = False

specs = [('bot','BEV — Bot (ground-truth) v/w','bev_bot_vw.png'),
         ('model','BEV — Model predicted waypoint rollouts','bev_model_wp.png'),
         ('both','BEV — Bot v/w (red) vs Model rollouts (blue)','bev_overlay.png')]
os.makedirs(SAVE_DIR, exist_ok=True); saved=[]
for which, title, fname in specs:
    fig, ax = plt.subplots(figsize=(11,9)); base(ax, title)
    if which in ('bot','both'):   draw_bot(ax)
    if which in ('model','both'): draw_model(ax)
    ax.legend(loc='upper left', framealpha=0.9); fig.tight_layout()
    path=os.path.join(SAVE_DIR, fname); fig.savefig(path, dpi=150, bbox_inches='tight')
    saved.append(path); plt.show()
print("Saved:"); [print("  ",p) for p in saved]

[DBG] recorded steps in seq            : 200
[DBG] total images available (IMAGE_DIR): 200
[DBG] goal_img_idx=102  goal_xy=(-0.3696031607061243, 16.314449982877868)
[DBG] stop knobs: GOAL_COST_THRESH=3.0 GOAL_IDX_TOL=5 MIN_PROGRESS_FRAC=0.25 CONFIRM_FRAMES=3
[DBG] loc_idx  first=11 last=5 min=2 max=12
[DBG] map coverage: max_loc 12/102 = 0.12
[DBG] loc_idx full path: [11, 8, 8, 11, 8, 8, 11, 11, 11, 11, 11, 9, 11, 11, 8, 7, 7, 7, 10, 10, 10, 10, 10, 10, 10, 8, 11, 11, 11, 11, 11, 10, 10, 9, 12, 12, 9, 9, 9, 9, 9, 9, 8, 8, 7, 7, 7, 7, 7, 7, 7, 6, 6, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 6, 5, 5, 2, 2, 2, 2, 2, 2, 2, 4, 4, 4, 4, 7, 7, 7, 3, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 11, 8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 6, 6, 6, 5, 5, 6, 6, 8, 8, 8, 8, 9, 9, 9, 9, 11, 9, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 9, 11, 11, 11, 11, 11, 11, 11, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 6, 8, 8, 8, 8, 8, 8, 8, 9, 10, 10, 10, 6, 8, 8, 6, 6, 6, 6, 9, 6, 6, 6, 7, 9, 9, 9, 9, 9, 5, 5, 5, 8, 8, 8

/tmp/ipykernel_220252/2235604830.py:103: UserWarning: Glyph 772 (\N{COMBINING MACRON}) missing from font(s) Liberation Sans.
  ax.legend(loc='upper left', framealpha=0.9); fig.tight_layout()
/tmp/ipykernel_220252/2235604830.py:104: UserWarning: Glyph 772 (\N{COMBINING MACRON}) missing from font(s) Liberation Sans.
  path=os.path.join(SAVE_DIR, fname); fig.savefig(path, dpi=150, bbox_inches='tight')


Saved:
   /home/anirudhadg/mast3r-nav-ati/data/mast3r-nav/data/recordings/traj_001/images//bev_images/bev_bot_vw.png
   /home/anirudhadg/mast3r-nav-ati/data/mast3r-nav/data/recordings/traj_001/images//bev_images/bev_model_wp.png
   /home/anirudhadg/mast3r-nav-ati/data/mast3r-nav/data/recordings/traj_001/images//bev_images/bev_overlay.png


[None, None, None]